In [1]:
from agents import (
    Agent,
    Runner,
    ModelSettings,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
)
from openai import OpenAI, AsyncOpenAI
from pydantic import BaseModel, Field
from typing import Literal
from datetime import datetime

set_tracing_disabled(True)

In [2]:
local_model = OpenAIChatCompletionsModel(
    model="qwen2.5-coder:3b",
    openai_client=AsyncOpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama"),
)

In [3]:
bad_agent = Agent(name="Bad Agent", instructions="You are helpful.", model=local_model)

# GOOD instructions — specific, structured, actionable
good_agent = Agent(
    model=local_model,
    name="Code Reviewer",
    instructions="""
    You are a senior Python code reviewer at a tech company.
    
    YOUR ROLE:
    - Review Python code for bugs, style issues, and performance problems.
    - Rate code quality from 1-10.
    - Suggest specific improvements with corrected code snippets.
    
    YOUR RULES:
    - Always check for: type hints, docstrings, error handling, edge cases.
    - Be constructive, not harsh.
    - If the code is good, say so — don't invent problems.
    - Keep reviews concise (under 200 words).
    
    OUTPUT FORMAT:
    Score: X/10
    Issues: [list]
    Suggestions: [list with code]
    """,
)

result = await Runner.run(
    good_agent,
    """
Review this code:

def add(x, y):
    return x + y
""",
)

print(result.final_output)

Score: 8/10
Issues: ['missing type hints and docstring']
Suggestions: ```python
from typing import Union

def add(x: Union[int, float], y: Union[int, float]) -> Union[int, float]:
    """Return the sum of x and y."""
    return x + y
```


In [4]:
creative_agent = Agent(
    model=local_model,
    name="Poet",
    instructions="You write creative, imaginative descriptions. Be vivid and poetic.",
    model_settings=ModelSettings(
        temperature=0.9,  # High = more creative/random
        max_tokens=500,  # Limit output length
    ),
)

# Precise agent — low temperature
precise_agent = Agent(
    model=local_model,
    name="Fact Checker",
    instructions="You provide precise, factual answers. No speculation. Cite sources if possible.",
    model_settings=ModelSettings(
        temperature=0.1,  # Low = more deterministic/precise
        max_tokens=300,
    ),
)

# Same question, different agents
question = "Describe the Badshahi Mosque in Lahore."

creative_result = await Runner.run(creative_agent, question)
precise_result = await Runner.run(precise_agent, question)

print(" CREATIVE:\n", creative_result.final_output[:300])
print("\n PRECISE:\n", precise_result.final_output[:300])

 CREATIVE:
 Stairways carved into intricate designs lead you deep into the Badshahi Mosque, a magnificent edifice that stands as a testament to Mughal architecture's grandeur and cultural heritage. The mosque is adorned with an array of majestic domes, each reaching towards the azure sky as if beckoning visitor

 PRECISE:
 The Badshahi Mosque is a significant Islamic architectural masterpiece located in Lahore, Pakistan. It was built by Mughal Emperor Aurangzeb between 1673 and 1680 as his personal mosque for the royal family. Here are some key points about the mosque:

1. Location: The mosque stands on the site of an


In [5]:
class PodcastReview(BaseModel):
    title: str = Field(description="Podcast title")
    host: str = Field(description="Podcast host(s)")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    genre: str = Field(
        description="Primary genre (e.g., AI Research, Machine Learning, AI News, AI Ethics)"
    )
    technical_level: str = Field(description="Beginner, Intermediate, or Advanced")
    summary: str = Field(description="One-sentence summary")
    recommendation: bool = Field(description="Would you recommend this?")


# Create agent with structured output
reviewer = Agent(
    model=local_model,
    name="AI Podcast Reviewer",
    instructions=(
        "You are an AI podcast critic. Analyze podcasts about artificial intelligence, "
        "machine learning, and related topics. Provide structured reviews and assess "
        "the technical depth and accessibility of each podcast."
    ),
    output_type=PodcastReview,
)

result = await Runner.run(reviewer, "Review the podcast 'Lex Fridman Podcast'")

# result.final_output is now a PodcastReview object, not a string!
review = result.final_output
print(f"Title:          {review.title}")
print(f"Host:           {review.host}")
print(f"Rating:         {review.rating}/10")
print(f"Genre:          {review.genre}")
print(f"Tech Level:     {review.technical_level}")
print(f"Summary:        {review.summary}")
print(f"Recommended:    {'Yes' if review.recommendation else 'No'}")

Title:          Lex Fridman Podcast
Host:           Lex Fridman
Rating:         4.5/10
Genre:          Science & Skepticism, Technology & Innovation, General Interest
Tech Level:     Intermediate to Advanced
Summary:        A conversational science pod with an in-depth delve into many technological topics with guests ranging from MIT professors to Google executives.
Recommended:    Yes


In [6]:
class ProductClassification(BaseModel):
    category: Literal["electronics", "clothing", "food", "books", "other"] = Field(
        description="Product category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgently does the customer need this?"
    )
    price_range: Literal["budget", "mid", "premium"] = Field(
        description="Estimated price range based on description"
    )
    search_query: str = Field(
        description="Optimized search query for the product catalog"
    )
    confidence: float = Field(description="Confidence score 0.0-1.0")


classifier = Agent(
    model=local_model,
    name="Product Classifier",
    instructions="""
    You classify customer product requests for an e-commerce platform.
    Analyze the customer's natural language description and extract:
    - What category the product falls into
    - How urgent the request seems
    - Estimated price range
    - An optimized search query
    - Your confidence in the classification
    
    Be accurate. If you're unsure, set confidence low.
    """,
    output_type=ProductClassification,
    model_settings=ModelSettings(temperature=0.1),
)

# Test with different customer requests
requests = [
    "I need a laptop for my son's school, nothing too expensive",
    "Looking for a birthday gift, maybe a nice novel",
    "URGENT: Need a phone charger, mine just broke!",
]

for req in requests:
    result = await Runner.run(classifier, req)
    c = result.final_output
    print(f"\nRequest: '{req}'")
    print(f"   Category: {c.category} | Urgency: {c.urgency} | Price: {c.price_range}")
    print(f"   Search: '{c.search_query}' | Confidence: {c.confidence:.0%}")


Request: 'I need a laptop for my son's school, nothing too expensive'
   Category: electronics | Urgency: low | Price: budget
   Search: 'laptop for school budget' | Confidence: 9000%

Request: 'Looking for a birthday gift, maybe a nice novel'
   Category: books | Urgency: low | Price: mid
   Search: 'birthday gifts novels' | Confidence: 9000%

Request: 'URGENT: Need a phone charger, mine just broke!'
   Category: electronics | Urgency: high | Price: mid
   Search: 'phone charger broken' | Confidence: 9000%


In [7]:
def dynamic_instructions(context, agent):
    """Instructions that change based on current state."""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M")
    return f"""
    You are a scheduling assistant.
    Current date/time: {current_time}
    
    Rules:
    - If it's morning (before 12pm), suggest productive tasks.
    - If it's afternoon, suggest meetings and collaborations.
    - If it's evening, suggest winding down activities.
    - Always be aware of the current time in your suggestions.
    """


scheduler = Agent(
    model=local_model,
    name="Smart Scheduler",
    instructions=dynamic_instructions,  # <-- Pass function, not string!
)

result = await Runner.run(scheduler, "What should I do right now?")
print(result.final_output)

Based on the current date/time (2026-04-09 11:49), you're currently in the morning. It's appropriate to suggest productive tasks for this period. Here are some ideas:

1. **Brainstorming**: Start a project or brainstorm for upcoming events, projects, or new initiatives.
2. **Reading and Learning**: Engage with professional or personal development materials that align with your interests or goals.
3. **Health Routine**: If you maintain a regular health routine, this is a good time to exercise or meditate.
4. **Efficient Work Tasks**: Identify high-priority tasks at work that require focus, such as deadline-driven assignments.

Remember, productivity starts with prioritizing and organizing your workload.


In [8]:
class TicketAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative", "angry"] = Field(
        description="Customer sentiment"
    )
    priority: Literal["P0-critical", "P1-high", "P2-medium", "P3-low"] = Field(
        description="Ticket priority"
    )
    department: Literal["billing", "technical", "sales", "general"] = Field(
        description="Which department should handle this"
    )
    summary: str = Field(description="One-line summary for the dashboard")
    suggested_response: str = Field(description="Draft first response to customer")


ticket_analyzer = Agent(
    model=local_model,
    name="Ticket Analyzer",
    instructions="""
    You are an AI support ticket analyzer for a SaaS company.
    
    PRIORITY RULES:
    - P0-critical: Service down, data loss, security breach
    - P1-high: Feature broken, payment issues, angry customer
    - P2-medium: Bug reports, feature requests, how-to questions
    - P3-low: General feedback, suggestions, non-urgent inquiries
    
    DEPARTMENT ROUTING:
    - billing: Payment, subscription, invoice, refund
    - technical: Bugs, errors, API issues, integrations
    - sales: Pricing, enterprise plans, demos, upgrades
    - general: Everything else
    
    Write the suggested_response in a professional, empathetic tone.
    """,
    output_type=TicketAnalysis,
    model_settings=ModelSettings(temperature=0.2),
)

# Simulate incoming tickets
tickets = [
    "Your API has been returning 500 errors for 2 hours. Our production is DOWN. Fix this NOW!",
    "Hi, I was charged twice this month. Can you please look into it? Thanks.",
    "Love the product! Any chance you'll add dark mode soon?",
]

for ticket in tickets:
    result = await Runner.run(ticket_analyzer, ticket)
    t = result.final_output
    print(f"\n{'=' * 60}")
    print(f"Ticket: {ticket[:60]}...")
    print(f"Sentiment: {t.sentiment}")
    print(f"Priority:  {t.priority}")
    print(f"Route to:  {t.department}")
    print(f"Summary:   {t.summary}")
    print(f"Response:  {t.suggested_response[:150]}...")


Ticket: Your API has been returning 500 errors for 2 hours. Our prod...
Sentiment: negative
Priority:  P0-critical
Route to:  technical
Summary:   API issue causing service disruption
Response:  I'm sorry to hear that our API has been experiencing issues, leading to a service outage. I understand the urgency of this situation and am working di...

Ticket: Hi, I was charged twice this month. Can you please look into...
Sentiment: neutral
Priority:  P1-high
Route to:  billing
Summary:   Customer has reported being charged twice for the same service.
Response:  Dear [Customer's Name], Thank you for reaching out to us. We apologize for any inconvenience caused by this issue. Our billing team will investigate t...

Ticket: Love the product! Any chance you'll add dark mode soon?...
Sentiment: positive
Priority:  P3-low
Route to:  general
Summary:   The user expresses their satisfaction with the product and requests a feature enhancement, specifically asking for dark mode. This is categorize